In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg16, VGG16_Weights
import numpy as np
from tqdm.auto import tqdm

In [ ]:
COMPETITION_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'manifest.csv' in files:
        # manifest.csv is usually inside the 'dataset' subfolder
        COMPETITION_ROOT = os.path.dirname(root) 
        DATASET_ROOT = root
        break

if COMPETITION_ROOT is None:
    raise FileNotFoundError("Could not find competition data in /kaggle/input/")

# Add the competition root to Python's path so it can find the 'utility' folder
if COMPETITION_ROOT not in sys.path:
    sys.path.append(COMPETITION_ROOT)

print(f"Competition Root: {COMPETITION_ROOT}")
print(f"Dataset Root: {DATASET_ROOT}")

from utility.data import HackathonDataset, combined_roll_from_bundle
from utility.metric import score_item, leaderboard_score
from utility.submission import bundle_to_rows, write_submission


In [ ]:
class PianoRollDataset(Dataset):
    def __init__(self, dataset_obj, split="train"):
        self.dataset = dataset_obj
        self.split = split
        if split == "train":
            self.triplets = self.dataset.train()
        elif split == "val":
            self.triplets = self.dataset.val()
        else:
            self.triplets = self.dataset.test()
            
    def _bundle_to_tensor(self, bundle):
        pitched = combined_roll_from_bundle(bundle)
        return torch.tensor(np.stack([pitched, bundle["drum"]]), dtype=torch.float32)

    def __len__(self): return len(self.triplets)

    def __getitem__(self, idx):
        t = self.triplets[idx]
        load_Y = self.split in ["train", "val"]
        data_dict = self.dataset.get(t.item_id, load_Y=load_Y)
        
        X = self._bundle_to_tensor(data_dict["X"])
        Z = self._bundle_to_tensor(data_dict["Z"])
        
        if load_Y:
            Y = self._bundle_to_tensor(data_dict["Y"])
            return X, Z, Y, t.item_id
        return X, Z, t.item_id

In [ ]:
class AdaIN(nn.Module):
    def forward(self, content, gamma, beta):
        b, c, h, w = content.size()
        c_std, c_mean = torch.std_mean(content, dim=[2, 3], keepdim=True)
        c_norm = (content - c_mean) / (c_std + 1e-5)
        return c_norm * gamma.view(b, c, 1, 1) + beta.view(b, c, 1, 1)

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, normalize=True):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, stride=2)]
        if normalize: layers.append(nn.InstanceNorm2d(out_c))
        layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class StyleTransferGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.content_enc = nn.Sequential(
            ConvBlock(2, 32), ConvBlock(32, 64), ConvBlock(64, 128), ConvBlock(128, 256)
        )
        self.style_enc = nn.Sequential(
            ConvBlock(2, 32, False), ConvBlock(32, 64, False), 
            ConvBlock(64, 128, False), ConvBlock(128, 256, False),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc_gamma = nn.Linear(256, 256)
        self.fc_beta = nn.Linear(256, 256)
        self.adain = AdaIN()
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.InstanceNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.InstanceNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.InstanceNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 2, 4, 2, 1), nn.Sigmoid()
        )

    def forward(self, x, z):
        c_feat = self.content_enc(x)
        s_feat = self.style_enc(z).view(-1, 256)
        adain_feat = self.adain(c_feat, self.fc_gamma(s_feat), self.fc_beta(s_feat))
        return self.decoder(adain_feat)

In [ ]:
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
        self.blocks = nn.ModuleList([vgg[:4], vgg[4:9], vgg[9:16]]).eval()
        for p in self.parameters(): p.requires_grad = False

    def forward(self, y_hat, y_tgt, z_style):
        def to_3c(t): return torch.cat([t, t[:, 1:2, :, :]], dim=1)
        x_hat, x_tgt, x_z = to_3c(y_hat), to_3c(y_tgt), to_3c(z_style)
        
        l_perc, l_style = 0.0, 0.0
        for block in self.blocks:
            x_hat, x_tgt, x_z = block(x_hat), block(x_tgt), block(x_z)
            l_perc += F.l1_loss(x_hat, x_tgt)
            b, c, h, w = x_hat.shape
            gram_hat = torch.bmm(x_hat.view(b, c, h*w), x_hat.view(b, c, h*w).transpose(1, 2)) / (c*h*w)
            gram_z = torch.bmm(x_z.view(b, c, h*w), x_z.view(b, c, h*w).transpose(1, 2)) / (c*h*w)
            l_style += F.mse_loss(gram_hat, gram_z)
        return l_perc, l_style

In [ ]:
def validate_model(model, dataset_obj, val_loader, device):
    model.eval()
    all_scores = []
    
    with torch.no_grad():
        for X_tensor, Z_tensor, _, item_ids in val_loader:
            X_tensor, Z_tensor = X_tensor.to(device), Z_tensor.to(device)
            Y_hat_tensor = model(X_tensor, Z_tensor)
            
            for i in range(len(item_ids)):
                item_id = item_ids[i]
                y_out = Y_hat_tensor[i].cpu().numpy()
                
                output_bundle = {
                    "pitched": np.expand_dims(y_out[0], axis=0),
                    "drum": y_out[1],
                    "track_ids": np.array([0], dtype=np.int32)
                }
                
                raw_data = dataset_obj.get(item_id, load_Y=False)
                content_roll = combined_roll_from_bundle(raw_data["X"])
                target_profile = raw_data["style_profile"]
                
                # Calculate scores
                item_score_dict = score_item(content_roll, output_bundle, target_profile)
                all_scores.append(item_score_dict)

    mean_score = leaderboard_score(all_scores)
    mean_cp = np.mean([d["CP"] for d in all_scores])
    mean_sf = np.mean([d["SF"] for d in all_scores])
    
    return mean_score, mean_cp, mean_sf

In [ ]:
def train_model(dataset_root, epochs=15, batch_size=8, lr=2e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")
    
    hack_data = HackathonDataset(dataset_root)
    train_ds = PianoRollDataset(hack_data, split="train")
    val_ds = PianoRollDataset(hack_data, split="val")
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=0)
    
    model = StyleTransferGenerator().to(device)
    vgg_loss = VGGPerceptualLoss().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    best_val_score = 0.0
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        loop = tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1}/{epochs}")
        
        for X, Z, Y, _ in loop:
            X, Z, Y = X.to(device), Z.to(device), Y.to(device)
            optimizer.zero_grad()
            Y_hat = model(X, Z)
            
            mse_loss = F.mse_loss(Y_hat, Y)
            perc_loss, style_loss = vgg_loss(Y_hat, Y, Z)
            loss = (10.0 * mse_loss) + (1.0 * perc_loss) + (5.0 * style_loss)
            
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            loop.set_postfix(loss=loss.item())
            
        avg_train_loss = epoch_loss / len(train_loader)
        val_score, val_cp, val_sf = validate_model(model, hack_data, val_loader, device)
        
        print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | "
              f"Val Score: {val_score:.4f} (CP: {val_cp:.4f}, SF: {val_sf:.4f})")
        
        if val_score > best_val_score:
            best_val_score = val_score
            torch.save(model.state_dict(), "best_model.pth")
            print("   -> 🌟 New Best Score! Saved best_model.pth")
            
    model.load_state_dict(torch.load("best_model.pth"))
    return model, hack_data

In [ ]:
def generate_submission(model, hack_data, output_csv="submission.csv"):
    device = next(model.parameters()).device
    model.eval()
    
    test_ds = PianoRollDataset(hack_data, split="test")
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
    
    all_rows = []
    print("\nGenerating Kaggle submission...")
    with torch.no_grad():
        for X, Z, item_id in tqdm(test_loader):
            X, Z = X.to(device), Z.to(device)
            Y_hat = model(X, Z)
            
            y_out = Y_hat.squeeze(0).cpu().numpy()
            bundle = {
                "pitched": np.expand_dims(y_out[0], axis=0),
                "drum": y_out[1],
                "track_ids": np.array([0], dtype=np.int32)
            }
            
            item_rows = list(bundle_to_rows(item_id[0], bundle))
            if len(item_rows) == 0:
                 all_rows.append({"item_id": item_id[0], "notes": ""}) 
            else:
                 all_rows.extend(item_rows)
            
    # Output saves directly to Kaggle's working directory /kaggle/working/
    write_submission(output_csv, all_rows)
    print(f"✅ Saved to {output_csv}. Submit this output file!")

In [ ]:
trained_model, dataset = train_model(DATASET_ROOT, epochs=5, batch_size=8, lr=2e-4)

In [ ]:
generate_submission(trained_model, dataset, output_csv="submission.csv")